# Module 09 — LLM Apps and APIs

**Offline educational demonstration.** Build a CLI FAQ application with optional API integration notes.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
p = Path('data')
if not p.exists(): p = Path('../data')
faq = pd.read_csv(p/'helpdesk_faq.csv')
labels = pd.read_csv(p/'evaluation_queries.csv')
print(faq[['doc_id','category','question']].head())


## Build and inspect a local TF-IDF retrieval index

In [ ]:
texts = (faq['question']+' '+faq['answer']).tolist()
vectorizer = TfidfVectorizer(stop_words='english', ngram_range=(1,2))
matrix = vectorizer.fit_transform(texts)
def search(query, k=3, category=None):
    candidates = faq.index if category is None else faq.index[faq['category'].eq(category)]
    if len(candidates)==0: return pd.DataFrame(columns=['doc_id','question','answer','score'])
    scores = cosine_similarity(vectorizer.transform([query]), matrix[candidates]).ravel()
    ix = np.argsort(-scores)[:k]
    result = faq.loc[candidates[ix], ['doc_id','question','answer']].copy()
    result['score'] = scores[ix]
    return result.reset_index(drop=True)
print(search('How can I track my delivery?'))


In [ ]:
def cli_answer(q):
    hit=search(q,k=1)
    if hit.empty or hit.iloc[0].score<0.1:return {'status':'escalate','answer':None}
    return {'status':'answered','answer':hit.iloc[0]['answer'],'source':hit.iloc[0]['doc_id']}
print(cli_answer('Where is my invoice?'))

## Reflection
Explain one failure mode, one measurement you would track, and how the demo differs from a production LLM system.